## Database Connection

Defines four upsert functions intended to be called by the orchestrator:

**Bronze (raw payloads)**
- `load_raw_ohlcv(symbol, payload)` — stores the raw OHLCV JSON, keyed by `(symbol, date)`
- `load_raw_overview(symbol, payload)` — stores the raw Overview JSON, keyed by `symbol`

**Silver (typed, deduplicated)**
- `load_ohlcv(df)` — upserts the OHLCV DataFrame into `market.ohlcv`
- `load_overview(df)` — upserts the Overview DataFrame into `market.overview`

All functions use `ON CONFLICT DO UPDATE` to make ingestion idempotent — running the same load twice produces the same end state.

### Setup

In [ ]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

### Connection

Credentials are read from the `.env` file. The `.env` must contain:
```
PG_USER=postgres
PG_PWD=your_password
PG_HOST=localhost
PG_PORT=5432
PG_DB=market
```

In [ ]:
load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['PG_USER']}:{os.environ['PG_PWD']}"
    f"@{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['PG_DB']}"
)

### Bronze

Bronze functions receive the raw payload dict (as returned by `ohlcv_bronze` and `overview_bronze`) and store it as JSONB.

- `load_raw_ohlcv` extracts `Last Refreshed` from the metadata as the partition key — one bronze row per (symbol, date).
- `load_raw_overview` uses `symbol` alone as the key — Overview is a snapshot, not a time series.

In [ ]:
upsert_raw_ohlcv = text("""
    INSERT INTO market.raw_ohlcv (symbol, date, payload)
    VALUES (:symbol, :date, CAST(:payload AS JSONB))
    ON CONFLICT (symbol, date) DO UPDATE SET
        payload = EXCLUDED.payload,
        extracted_at = NOW()
""")


def load_raw_ohlcv(symbol: str, payload: dict) -> None:
    date = payload["Meta Data"]["3. Last Refreshed"]
    with engine.begin() as conn:
        conn.execute(
            upsert_raw_ohlcv,
            {"symbol": symbol, "date": date, "payload": json.dumps(payload)},
        )

In [ ]:
upsert_raw_overview = text("""
    INSERT INTO market.raw_overview (symbol, payload)
    VALUES (:symbol, CAST(:payload AS JSONB))
    ON CONFLICT (symbol) DO UPDATE SET
        payload = EXCLUDED.payload,
        extracted_at = NOW()
""")


def load_raw_overview(symbol: str, payload: dict) -> None:
    with engine.begin() as conn:
        conn.execute(
            upsert_raw_overview,
            {"symbol": symbol, "payload": json.dumps(payload)},
        )

### Silver

Silver functions receive the cleaned DataFrame (as returned by `ohlcv_silver` and `overview_silver`) and upsert it into the typed tables.

- `load_ohlcv` keyed by `(symbol, date)` — re-runs for the same trading day overwrite.
- `load_overview` keyed by `(symbol, snapshot_date)` where `snapshot_date = CURRENT_DATE` — keeps a daily history of fundamental changes, since values like market cap shift even when the underlying `LatestQuarter` hasn't changed.

In [ ]:
upsert_ohlcv = text("""
    INSERT INTO market.ohlcv (symbol, date, open, high, low, close, volume)
    VALUES (:symbol, :date, :open, :high, :low, :close, :volume)
    ON CONFLICT (symbol, date) DO UPDATE SET
        open = EXCLUDED.open,
        high = EXCLUDED.high,
        low = EXCLUDED.low,
        close = EXCLUDED.close,
        volume = EXCLUDED.volume,
        loaded_at = NOW()
""")


def load_ohlcv(df: pd.DataFrame) -> int:
    records = df.to_dict(orient="records")
    with engine.begin() as conn:
        conn.execute(upsert_ohlcv, records)
    return len(records)

The Overview silver upsert renames columns from the API's PascalCase to snake_case and coerces numeric fields. The `snapshot_date` is added inside the function to ensure a consistent value across all rows of the same run.

In [ ]:
overview_renames = {
    "Symbol": "symbol",
    "AssetType": "asset_type",
    "Name": "name",
    "Description": "description",
    "CIK": "cik",
    "Exchange": "exchange",
    "Currency": "currency",
    "Country": "country",
    "Sector": "sector",
    "Industry": "industry",
    "Address": "address",
    "OfficialSite": "official_site",
    "FiscalYearEnd": "fiscal_year_end",
    "LatestQuarter": "latest_quarter",
    "MarketCapitalization": "market_capitalization",
    "EBITDA": "ebitda",
    "PERatio": "pe_ratio",
    "PEGRatio": "peg_ratio",
    "BookValue": "book_value",
    "DividendPerShare": "dividend_per_share",
    "DividendYield": "dividend_yield",
    "EPS": "eps",
    "RevenuePerShareTTM": "revenue_per_share_ttm",
    "ProfitMargin": "profit_margin",
    "OperatingMarginTTM": "operating_margin_ttm",
    "ReturnOnAssetsTTM": "return_on_assets_ttm",
    "ReturnOnEquityTTM": "return_on_equity_ttm",
    "RevenueTTM": "revenue_ttm",
    "GrossProfitTTM": "gross_profit_ttm",
    "DilutedEPSTTM": "diluted_eps_ttm",
    "QuarterlyEarningsGrowthYOY": "quarterly_earnings_growth_yoy",
    "QuarterlyRevenueGrowthYOY": "quarterly_revenue_growth_yoy",
    "AnalystTargetPrice": "analyst_target_price",
    "AnalystRatingStrongBuy": "analyst_rating_strong_buy",
    "AnalystRatingBuy": "analyst_rating_buy",
    "AnalystRatingHold": "analyst_rating_hold",
    "AnalystRatingSell": "analyst_rating_sell",
    "AnalystRatingStrongSell": "analyst_rating_strong_sell",
    "TrailingPE": "trailing_pe",
    "ForwardPE": "forward_pe",
    "PriceToSalesRatioTTM": "price_to_sales_ratio_ttm",
    "PriceToBookRatio": "price_to_book_ratio",
    "EVToRevenue": "ev_to_revenue",
    "EVToEBITDA": "ev_to_ebitda",
    "Beta": "beta",
    "52WeekHigh": "week_52_high",
    "52WeekLow": "week_52_low",
    "50DayMovingAverage": "moving_average_50_day",
    "200DayMovingAverage": "moving_average_200_day",
    "SharesOutstanding": "shares_outstanding",
    "SharesFloat": "shares_float",
    "PercentInsiders": "percent_insiders",
    "PercentInstitutions": "percent_institutions",
    "DividendDate": "dividend_date",
    "ExDividendDate": "ex_dividend_date",
}

In [ ]:
def load_overview(df: pd.DataFrame) -> int:
    df = df.rename(columns=overview_renames).copy()
    df["snapshot_date"] = pd.Timestamp.now().date()

    # Replace empty strings and 'None' with NaN so they become NULL in Postgres
    df = df.replace({"": None, "None": None, "-": None})

    # Only keep columns that exist in the target table
    expected_cols = ["symbol", "snapshot_date"] + [v for k, v in overview_renames.items() if v != "symbol"]
    df = df.reindex(columns=expected_cols)

    upsert_overview = text(f"""
        INSERT INTO market.overview ({', '.join(expected_cols)})
        VALUES ({', '.join([':' + c for c in expected_cols])})
        ON CONFLICT (symbol, snapshot_date) DO UPDATE SET
            { ', '.join([f'{c} = EXCLUDED.{c}' for c in expected_cols if c not in ['symbol', 'snapshot_date']]) },
            loaded_at = NOW()
    """)

    records = df.to_dict(orient="records")
    with engine.begin() as conn:
        conn.execute(upsert_overview, records)
    return len(records)